In [ ]:
# Import the libraries required for data handling, modeling, and evaluation.
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier

In [2]:
# Set a fixed random seed to make model training and validation reproducible.
seed = 1314525

In [ ]:
# Load the training and external validation datasets from CSV files.
train_df = pd.read_csv('data_train.csv')

In [ ]:
X_train = train_df.iloc[:, 2:]
y_train = train_df.iloc[:, 1]

In [ ]:
# Instantiate the classifier using the selected or predefined hyperparameters.
clf = TabPFNClassifier(random_state = seed)
# Fit the model on the training cohort.
clf.fit(X_train, y_train)
# Generate predicted probabilities for ROC-AUC evaluation.
pro_train = clf.predict_proba(X_train)[:, 1]
# Report model performance metrics for this cohort.
roc_auc_train = roc_auc_score(y_train, pro_train)
# Generate class labels for accuracy and related classification metrics.
accuracy_train = accuracy_score(y_train, clf.predict(X_train))
print(f'Training Set - AUC: {roc_auc_train:.3f}, Accuracy: {accuracy_train:.3f}')

In [ ]:
df_train = pd.DataFrame({
    'ID': train_df['ID'],
    'True': y_train,
    'Pre': pro_train
})
# Export predicted labels so downstream confusion-matrix analyses can reuse them.
df_train.to_csv('TabPFN_train.csv', index=False)

In [ ]:
test_files = ['data_test1.csv', 'data_test2.csv', 'data_test3.csv']

for test_file in test_files:
    # Load the training and external validation datasets from CSV files.
    test_df = pd.read_csv(test_file)
    
    X_test = test_df.iloc[:, 2:]
    y_test = test_df.iloc[:, 1]

    # Generate predicted probabilities for ROC-AUC evaluation.
    pro_test = clf.predict_proba(X_test)[:, 1]

    # Report model performance metrics for this cohort.
    roc_auc_test = roc_auc_score(y_test, pro_test)
    # Generate class labels for accuracy and related classification metrics.
    accuracy_test = accuracy_score(y_test, clf.predict(X_test))

    print(f'{test_file} - AUC: {roc_auc_test:.3f}, Accuracy: {accuracy_test:.3f}')

    df_test = pd.DataFrame({
        'ID': test_df['ID'],
        'True': y_test,
        'Pre': pro_test
    })
    
    # Export predicted labels so downstream confusion-matrix analyses can reuse them.
    df_test.to_csv(f'TabPFN_{test_file.split("/")[-1].split(".")[0]}_predictions.csv', index=False)

In [ ]:
# Import the libraries required for data handling, modeling, and evaluation.
import joblib
# Save the trained model for later reuse.
joblib.dump(clf, 'saved_model/TabPFN.pkl')